<div align="right"><a href="L35-SpektraleSchaetzverfahren.ipynb">Switch to German language 🇩🇪</a></div>

Die folgenden Übungen sind Teil des Buches
* K.-D. Kammeyer, K. Kroschel: "Digitale Signalverarbeitung - Filterung und Spektralanalyse", Springer Vieweg, 2022.  [[KK22]](https://doi.org/10.1007/978-3-663-09805-8)

---

This Jupyter Notebook can be opened in [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb) by clicking on the icon on the right, but this notebook is displayed best when opened locally using [Jupyter Notebook](https://jupyter.org/) (e.g. as part of the [Anaconda](https://www.anaconda.com/) distribution). <a href="https://colab.research.google.com/github/Stefan-Goe/dsp-book/blob/main/python/S35-SpectralEstimators.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Notebook in Google Colab"></a>

---

<!--NAVIGATION-->
<div align="center">

Previous: [S34 Burg-Algorithm](./S34-Burg-Algorithm.ipynb) | [Exercise 35](./E35-SpectralEstimators.ipynb) | [Contents / Overview](./00-Python-Notebooks-Overview-EN.ipynb)

</div>

---


## Exercise: Comparison of the Estimation Algorithms

In this exercise, three algorithms for estimating the coefficients of an AR($N$) model (the **autocorrelation method**, **autocovariance method**, and **Burg algorithm**) are compared with respect to their **bias** and **variance**.

In addition, a method based on the autocorrelation approach is examined, which applies a **Hamming window** instead of a rectangular window to evaluate the data (Python function: `numpy.hamming(·)`).

---

### Given

An (all-pole) transfer system

$$
H(z) = \frac{1}{A(z)}
$$

with

$$
\begin{aligned}
A(z) =;& 1 - 2.2346 z^{-1} + 2.3921 z^{-2} - 2.0324 z^{-3} \
&+ 1.2865 z^{-4} - 0.4954 z^{-5} + 0.2322 z^{-6} \
&- 0.2554 z^{-7} - 0.1848 z^{-8}
\end{aligned}
$$

of order $m = 8$, which is excited by a stationary, zero-mean, white noise process $N(k)$ with power

$$
\sigma_{\pi N}^2 = 1 .
$$

<div style="text-align:center">
  <img src="./pngs/l15sys.eps" width="70%">
</div>

---

## Tasks

### a)

First, compute the **pole-zero plot** of the system. In Python, this can be done using functions from `scipy.signal` (e.g. by computing poles and zeros explicitly and plotting them).

For a more detailed analysis, determine the exact **PARCOR coefficients**

$$
\gamma_i, \quad i = 1 \ldots 8 .
$$

To do this, compute the **unbiased autocorrelation sequence** $r_{\pi XX}(\kappa)$ using the impulse response of the system and solve the exact **Yule–Walker equation** (Eq. 9.3.17) by applying the **Levinson–Durbin recursion**, for example via implementations available in Python libraries such as `scipy` or `statsmodels`.

---

### b)

Estimate the **power density spectrum (PDS)** of the process $X(k)$ using an AR($8$) model and the estimation methods described above.

Use **1000 realizations** of the stochastic process $X(k)$, where each realization consists of

$$
N = 64
$$

samples.

To account for the transient response of the system $H(z)$, generate at least

$$
2^{11} + 64
$$

samples of the excitation process $N(k)$. Filter the white noise using a digital filter implementation, for example with `scipy.signal.lfilter`.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from helper_functions import lyw, lcov, lburg_algo
# BEGIN SOLUTION
# ##############################################################################
# ##  Solution: Spectral Estimation Methods                                   ##
# ##############################################################################

MA = np.array([1.0])
AR = np.array([1.0000, -2.2346, 2.3921, -2.0324, 1.2865, -0.4954, 0.2322, -0.2554, 0.1848])
NFFT = 2**6
n = len(AR) - 1

N = 64
N_vor = 2**11
mc_run = 2000  # Set to ~200 for faster testing

# --- Part A: Transient Response & Pole-Zero Plot ---

plt.figure(figsize=(6, 6))
z, p, k = signal.tf2zpk(MA, AR)
plt.scatter(np.real(p), np.imag(p), marker='x', color='red', label='Poles')

# Draw unit circle
uc = plt.Circle((0,0), 1, fill=False, color='black', linestyle='--')
plt.gca().add_patch(uc)

plt.grid(True)
plt.title(f'Poles of the AR({n}) Model')
plt.xlabel('Real Part')
plt.ylabel('Imaginary Part')
plt.axis('equal')
plt.show()

# Calculate exact PARCOR coefficients (via impulse response)
t, h_arma = signal.dimpulse((MA, AR, 1), n=512)
h_arma = h_arma[0].flatten()
rxx_all = np.correlate(h_arma, h_arma, mode='full')
mid = len(rxx_all) // 2
rxx = rxx_all[mid:mid+n+1]

# Use Levinson-Durbin for exact PARCOR
_, g_exact, _ = lyw(h_arma, n, lev=1)
print("Exact PARCOR Coefficients:")
for val in g_exact:
    print(f"{val:1.4f}")

# --- Part B: Monte Carlo Simulation ---

Omega_norm = np.linspace(0, 1, NFFT, endpoint=False)
_, h_ex = signal.freqz(MA, AR, worN=NFFT, whole=True)
Sxx_ex = np.abs(h_ex)**2

for run_idx in range(1, 5):
    Sxx_ar = np.zeros((mc_run, NFFT))
    Coeff = np.zeros((mc_run, len(AR)))
    
    print(f"Calculating Method {run_idx}...")
    
    for r in range(mc_run):
        noise = np.random.randn(20 * N + N_vor)
        x_all = signal.lfilter(MA, AR, noise)
        x = x_all[N_vor : N_vor + N + 1]
        
        if run_idx == 1:
            ARc, _, _ = lyw(x, n, 1)
            method_str = 'YW'
        elif run_idx == 2:
            x_ham = x * np.hamming(len(x))
            ARc, _, _ = lyw(x_ham, n, 1)
            method_str = 'YW-Ham.'
        elif run_idx == 3:
            ARc = lcov(x, n)
            method_str = 'Covar.'
        elif run_idx == 4:
            ARc, _ = lburg_algo(x, len(x), n)
            method_str = 'Burg'
            
        Coeff[r, :] = ARc
        _, h_est = signal.freqz([1], ARc, worN=NFFT, whole=True)
        Sxx_ar[r, :] = np.abs(h_est)**2

    Coeff_mean = np.mean(Coeff, axis=0)
    Coeff_std = np.std(Coeff, axis=0)
    C2sig_o = Coeff_mean + 2 * Coeff_std
    C2sig_u = Coeff_mean - 2 * Coeff_std
    Coeff_mse = np.mean(np.abs(AR - Coeff_mean)**2) # Mean Squared Error
    
    Sxx_mean = np.mean(Sxx_ar, axis=0)
    Sxx_2sigma = Sxx_mean + 2 * np.std(Sxx_ar, axis=0)

    # --- Plot 1: Coefficients ---
    plt.figure()
    indices = np.arange(n + 1)
    plt.stem(indices, AR, linefmt='r-', markerfmt='ro', basefmt=" ", label='Exact')
    plt.stem(indices, Coeff_mean, linefmt='b-', markerfmt='bx', basefmt=" ", 
             label=f'Mean / MSE={Coeff_mse:1.2e}')
    
    # 2-Sigma vertical lines
    for i in range(len(indices)):
        plt.plot([indices[i], indices[i]], [C2sig_u[i], C2sig_o[i]], 'g-')
    plt.plot(indices, C2sig_o, 'go', fillstyle='full', label=f'2σ-Bound / σ={np.mean(Coeff_std):1.2f}')
    plt.plot(indices, C2sig_u, 'go', fillstyle='full')
    
    plt.title(f'{method_str}-AR({n}) Coefficients; N={N}')
    plt.xlabel('i')
    plt.ylabel('Coefficients')
    plt.grid(True)
    plt.ylim([-4, 4])
    plt.legend()
    plt.show()

    # --- Plot 2: Power Spectral Density (PSD) ---
    plt.figure()
    plt.plot(Omega_norm, 10 * np.log10(Sxx_ex / np.max(Sxx_ex)), 'r--', label='Exact')
    plt.plot(Omega_norm, 10 * np.log10(Sxx_mean / np.max(Sxx_ex)), 'b-', label='Mean')
    plt.plot(Omega_norm, 10 * np.log10(Sxx_2sigma / np.max(Sxx_ex)), 'g-.', label='2σ-Bound')
    
    plt.ylabel(r'$S_{xx}(e^{j\Omega})$ in dB')
    plt.xlabel(r'$\Omega/2\pi$')
    plt.title(f'PSD of {method_str}-AR({n}); N={N}')
    plt.legend()
    plt.grid(True)
    plt.show()
# END SOLUTION

## Solution: Comparison of the Estimation Algorithms

### a)

The figure shows the pole-zero plot of the system $H(z)$. It is a real-valued system, where a pole pair at

$$
\Omega / 2\pi \approx 0.05
$$

is located very close to the unit circle.

For the underlying AR(8) process, the following PARCOR coefficients result when the exact Yule–Walker equation is used:

$$
\gamma_1 = 0.9785,;
\gamma_2 = -0.9279,;
\gamma_3 = 0.1531,;
\gamma_4 = -0.6974,
$$

$$
\gamma_5 = 0.1815,;
\gamma_6 = -0.1563,;
\gamma_7 = -0.1631,;
\gamma_8 = -0.1848 .
$$

Note that the first two PARCOR coefficients have values close to one and therefore dominate the behavior of the AR process. This AR process corresponds to a typical speech process.

<div style="text-align:center">
  <img src="./pngs/ldsv9a04_a1.png">
</div>

---

### b)

In the following, the essential properties of the known spectral estimation algorithms are summarized. A sample interval of length $N$ of the process $X(k)$ is assumed. For the discussion, the **power of the prediction error** is considered.

Because of the inverse relationship between an AR($n$) model and a predictor, minimizing the prediction error power is equivalent to determining the parameters of an AR($n$) process. In other words, the process $X(k)$ is assigned a predictor whose coefficients are chosen such that the prediction error power is minimized and equals the underlying white noise process $N(k)$.

---

#### Yule–Walker / Autocorrelation Method

* Process $X(k)$ is assumed to vanish outside the sample interval (rectangular window)
* The setting time of the predictor is included in the minimization
* The **biased** autocorrelation sequence $\hat{r}_{\pi XX}(\kappa)$ is used
* The solution is based on a Hermitian Toeplitz matrix
* The estimation is **biased**, but the AR model is always **minimal phase**

---

#### Autocovariance Method

* Process $X(k)$ is assumed to be undefined outside the sample interval
* The setting time of the predictor is **not** included in the minimization
* The **unbiased** autocorrelation sequence $r_{\pi XX}(\kappa)$ is used
* No Toeplitz matrix exists
* The estimation is **unbiased**, but the AR model is not necessarily **minimal phase**

---

#### Burg Algorithm

* Joint minimization of forward and backward prediction error powers
* Process $X(k)$ is treated as undefined outside the sample interval
* The setting time of the predictor is included in the minimization
* The estimation is **unbiased** and the AR model is always **minimal phase**

---

#### Autocorrelation Method with Hamming Window

Instead of applying a rectangular window to the process $X(k)$, a Hamming window of the same length $N$ is applied.

Rather than using the correlation of two rectangular windows—which results in a Bartlett window—to compute the biased autocorrelation sequence, the correlation of two Hamming windows is used. This introduces a small bias for autocorrelation values $\hat{r}_{\pi XX}(\kappa)$ at small $\kappa$, while larger $\kappa$ values receive stronger weighting.

To illustrate this behavior, the unbiased correlation results of both windows are plotted. This method is frequently used in speech signal analysis, since speech signals typically exhibit a dominant first PARCOR coefficient, which is mainly determined by $\hat{r}*{\pi XX}(0)$ and $\hat{r}*{\pi XX}(1)$.

<div style="text-align:center">
  <img src="./pngs/ldsv9a04fenster.png">
</div>

---

The figures on the left-hand side show the estimated coefficients, while the figures on the right-hand side show the corresponding power density spectra.

<div style="text-align:center">
  <img src="./pngs/ldsv9a04_b11.png">
  <img src="./pngs/ldsv9a04_b12.png">
</div>

The results based on the autocorrelation method (“YW-AR(8)”) confirm the presence of bias in this estimation. For example, some exact coefficients such as $a_3$ lie outside the $2\sigma$ boundaries. This behavior is also reflected in the relatively high prediction error power

$$
E{e^2} = 1.18 .
$$

The power density spectrum is poorly approximated, and the critical pole at $\Omega / 2\pi \approx 0.05$ is incorrectly modeled.

---

<div style="text-align:center">
  <img src="./pngs/ldsv9a04_b21.png">
  <img src="./pngs/ldsv9a04_b22.png">
</div>

Using the autocorrelation method modified with a Hamming window (“YW-HAM-AR(8)”), a significant reduction of bias is observed for processes with large PARCOR coefficients $\gamma_1$ and $\gamma_2$.

For instance, coefficient $a_3$, which previously lay outside the $2\sigma$ boundaries, now lies inside them. The mean prediction error power decreases to

$$
E{e^2} = 0.25 .
$$

A drawback of the Hamming weighting is an increased standard deviation $\sigma$. The estimated power density spectrum shows a clear improvement when the Hamming window is used.

---

<div style="text-align:center">
  <img src="./pngs/ldsv9a04_b31.png">
  <img src="./pngs/ldsv9a04_b32.png">
</div>

An unbiased estimation of the coefficients is achieved using the covariance method (“cov-AR(8)”). The small deviations between the exact and mean values are due to the finite number of realizations ($1000$). The standard deviation behaves similarly to the modified autocorrelation method.

Thus, the covariance method represents an attractive alternative for spectral estimation, as confirmed by the approximation of the power density spectrum. Its main drawback is the lack of guaranteed minimal-phase behavior.

---

<div style="text-align:center">
  <img src="./pngs/ldsv9a04_b41.png">
  <img src="./pngs/ldsv9a04_b42.png">
</div>

The figures above show the results obtained with the Burg algorithm (“Burg-AR(8)”). The joint minimization of forward and backward prediction errors proves to be the most suitable approach. The absence of bias is clearly visible, and the power density spectrum is approximated almost as well as with the covariance method.

Regarding estimation variance, the Burg algorithm behaves similarly to the covariance and modified autocorrelation methods for the example considered here.
